In [ ]:
# Install qcirclab from the repository.
# In a local environment you may prefer: pip install -e /path/to/qcirclab_repo
!pip install -q git+https://github.com/2forts/qcirclab_repo.git

In [ ]:
import time
import math
import random
from collections import deque, Counter

import numpy as np

from qcirclab import (
    Circuit,
    circuit_metrics,
    print_metrics,
    append_operation,
    circuit_without_measurements,
    circuit_unitary,
    equal_up_to_global_phase,
)

import qcirclab.gates as qg

from qcirclab.passes import (
    AnalysisPass,
    TransformationPass,
    PassPipeline,
)

# Subsection 8.2.4 **Implementing global passes with qcirclab**

In [ ]:
class GlobalCostAnalysis(AnalysisPass):
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def run(self, qc: Circuit) -> Circuit:
        metrics = circuit_metrics(qc)
        depth = metrics["depth"]
        twoq_gates = metrics["two_qubit_gates"]
        cost = self.alpha * depth + self.beta * twoq_gates
        self.property_set["global_cost"] = cost
        self.property_set["global_metrics"] = metrics
        return qc

In [ ]:
SELF_INVERSE = {"h", "x", "y", "z", "cx", "cz", "swap"}


def same_location(op1, op2):
    return (
        op1.name == op2.name
        and tuple(op1.targets) == tuple(op2.targets)
        and tuple(op1.controls) == tuple(op2.controls)
        and op1.condition == op2.condition
    )


def cancel_adjacent_gates(qc: Circuit) -> Circuit:
    stack = []

    for op in qc.operations:
        if op.name == "barrier":
            continue

        if stack:
            prev = stack[-1]

            if (
                op.name in SELF_INVERSE
                and prev.name in SELF_INVERSE
                and same_location(prev, op)
            ):
                stack.pop()
                continue

            if (
                op.name in {"rx", "ry", "rz"}
                and prev.name == op.name
                and tuple(prev.targets) == tuple(op.targets)
                and tuple(prev.controls) == tuple(op.controls)
                and prev.params
                and op.params
                and abs(prev.params[0] + op.params[0]) < 1e-12
            ):
                stack.pop()
                continue

        stack.append(op)

    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cancelled")
    for op in stack:
        append_operation(out, op)
    return out


def propose_move(qc: Circuit) -> Circuit:
    return cancel_adjacent_gates(qc)

In [ ]:
class SimulatedAnnealingStep(TransformationPass):
    def __init__(self, alpha=1.0, beta=1.0, temperature=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.temperature = temperature

    def _cost(self, qc: Circuit):
        metrics = circuit_metrics(qc)
        return self.alpha * metrics["depth"] + self.beta * metrics["two_qubit_gates"]

    def run(self, qc: Circuit) -> Circuit:
        current_cost = self._cost(qc)
        candidate = propose_move(qc)
        candidate_cost = self._cost(candidate)

        if candidate_cost < current_cost:
            return candidate

        delta = candidate_cost - current_cost
        prob = math.exp(-delta / max(self.temperature, 1e-8))
        if random.random() < prob:
            return candidate
        return qc

# Shared basis-decomposition helper

In [ ]:
def decompose_to_rx_rz_cx(qc: Circuit) -> Circuit:
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_basis")

    for op in qc.operations:
        if op.name in {"barrier", "measure", "reset"}:
            append_operation(out, op)

        elif op.name == "h":
            q = op.targets[0]
            out.rz(np.pi/2, q)
            out.rx(np.pi/2, q)
            out.rz(np.pi/2, q)

        elif op.name == "x":
            out.rx(np.pi, op.targets[0])

        elif op.name == "z":
            out.rz(np.pi, op.targets[0])

        elif op.name == "s":
            out.rz(np.pi/2, op.targets[0])

        elif op.name == "t":
            out.rz(np.pi/4, op.targets[0])

        elif op.name == "tdg":
            out.rz(-np.pi/4, op.targets[0])

        elif op.name == "ry":
            q = op.targets[0]
            theta = op.params[0]
            out.rz(np.pi/2, q)
            out.rx(theta, q)
            out.rz(-np.pi/2, q)

        elif op.name == "cz":
            c = op.controls[0]
            t = op.targets[0]
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)
            out.cx(c, t)
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)

        elif op.name in {"rx", "rz", "cx"}:
            append_operation(out, op)

        else:
            append_operation(out, op)

    return out

In [ ]:
def generate_candidates(qc: Circuit):
    basis = decompose_to_rx_rz_cx(qc)

    return [
        ("original", qc),
        ("cancel_adjacent", cancel_adjacent_gates(qc)),
        ("basis_rx_rz_cx", basis),
        ("basis_then_cancel", cancel_adjacent_gates(basis)),
    ]


class BestGlobalCandidate(TransformationPass):
    def __init__(self, alpha=1.0, beta=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def _cost(self, qc: Circuit):
        metrics = circuit_metrics(qc)
        return (
            self.alpha * metrics["depth"]
            + self.beta * metrics["two_qubit_gates"]
        )

    def run(self, qc: Circuit) -> Circuit:
        candidates = generate_candidates(qc)

        scored = []
        for name, cand in candidates:
            cost = self._cost(cand)
            metrics = circuit_metrics(cand)
            scored.append((cost, name, cand, metrics))

        best_cost, best_name, best_circuit, best_metrics = min(
            scored,
            key=lambda x: x[0],
        )

        self.property_set["candidate_costs"] = [
            {
                "name": name,
                "cost": cost,
                "metrics": metrics,
            }
            for cost, name, _, metrics in scored
        ]
        self.property_set["best_candidate"] = best_name
        self.property_set["best_candidate_cost"] = best_cost

        return best_circuit

In [ ]:
def encode_circuit(qc: Circuit):
    m = circuit_metrics(qc)
    return np.array([
        m["operations"],
        m["depth"],
        m["two_qubit_gates"],
        m["multi_qubit_gates"],
    ], dtype=float)


class SimpleLinearModel:
    def __init__(self, weights=(0.2, 1.0, 1.5, 3.0)):
        self.weights = np.asarray(weights, dtype=float)

    def predict(self, features):
        return float(np.dot(self.weights, features))


class LearnedGlobalMove(TransformationPass):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def run(self, qc: Circuit) -> Circuit:
        candidates = generate_candidates(qc)

        scored = []
        for name, cand in candidates:
            features = encode_circuit(cand)
            score = self.model.predict(features)
            scored.append((score, name, cand))

        best_score, best_name, best_circuit = min(scored, key=lambda x: x[0])

        self.property_set["learned_move_score"] = best_score
        self.property_set["learned_move"] = best_name

        return best_circuit

In [ ]:
pm = PassPipeline([
    BestGlobalCandidate(alpha=1.0, beta=2.0),
    GlobalCostAnalysis(alpha=1.0, beta=2.0),
])

qc = Circuit(2)
qc.h(0)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 1)
qc.rz(0.3, 1)
qc.rz(-0.3, 1)
qc.x(0)
qc.x(0)
qc.cz(0, 1)
qc.cz(0, 1)

new_qc = pm.run(qc)

print("Original:")
print(qc.draw())
print_metrics("Original", qc)

print("After global candidate selection:")
print(new_qc.draw())
print_metrics("Selected circuit", new_qc)

print("Property set:")
print(pm.property_set)

# Subsection 8.3.5 **Custom rewrite rules and equivalence checking**

In [ ]:
class SimpleEquivalenceLibrary:
    def __init__(self):
        self._rules = {}

    def add_equivalence(self, gate_name, builder):
        self._rules.setdefault(gate_name, []).append(builder)

    def get_entry(self, gate_name):
        return self._rules.get(gate_name, [])


def cz_to_hcxh(c, t):
    decomp = Circuit(2)
    decomp.h(1)
    decomp.cx(0, 1)
    decomp.h(1)
    return decomp


equiv_lib = SimpleEquivalenceLibrary()
equiv_lib.add_equivalence("cz", cz_to_hcxh)


class CZToCXRewrite(TransformationPass):
    def __init__(self, equiv_lib):
        super().__init__()
        self.equiv_lib = equiv_lib

    def run(self, qc: Circuit) -> Circuit:
        out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cz_rewritten")

        for op in qc.operations:
            if op.name == "cz":
                c = op.controls[0]
                t = op.targets[0]
                out.h(t)
                out.cx(c, t)
                out.h(t)
            else:
                append_operation(out, op)

        return out

In [ ]:
qc_cz = Circuit(2)
qc_cz.h(0)
qc_cz.cz(0, 1)
qc_cz.rz(0.2, 1)

rewrite = CZToCXRewrite(equiv_lib)
qc_rewritten = rewrite.run(qc_cz)

print("Original:")
print(qc_cz.draw())
print_metrics("Original", qc_cz)

print("Rewritten:")
print(qc_rewritten.draw())
print_metrics("Rewritten", qc_rewritten)

print(
    "Equivalent up to global phase:",
    equal_up_to_global_phase(circuit_unitary(qc_cz), circuit_unitary(qc_rewritten))
)

# Subsection 8.4.4 **Timing-aware profiling with qcriclab utilities**

In [ ]:
gate_durations_ns = {
    "h": 35,
    "x": 35,
    "rx": 35,
    "ry": 35,
    "rz": 0,
    "s": 0,
    "t": 0,
    "tdg": 0,
    "cx": 300,
    "cz": 300,
    "swap": 900,
    "measure": 1000,
}


def schedule_asap(qc: Circuit, durations=gate_durations_ns):
    qtime = [0.0] * qc.n_qubits
    ctime = [0.0] * qc.n_clbits
    rows = []

    for op in qc.operations:
        if op.name == "barrier":
            m = max(qtime + ctime) if (qtime or ctime) else 0.0
            qtime = [m] * qc.n_qubits
            ctime = [m] * qc.n_clbits
            rows.append((op.name, (), (), m, m))
            continue

        used_q = set(op.targets) | set(op.controls)
        used_c = set(getattr(op, "ctargets", ()))
        if getattr(op, "condition", None) is not None:
            used_c.add(op.condition.bit)

        start = 0.0
        if used_q:
            start = max(start, max(qtime[q] for q in used_q))
        if used_c:
            start = max(start, max(ctime[c] for c in used_c))

        duration = durations.get(op.name, 50)
        finish = start + duration

        for q in used_q:
            qtime[q] = finish
        for c in used_c:
            ctime[c] = finish

        rows.append((op.name, tuple(sorted(used_q)), tuple(sorted(used_c)), start, finish))

    return rows, max(qtime + ctime) if (qtime or ctime) else 0.0


def print_schedule(rows):
    for name, qs, cs, start, finish in rows:
        print(f"{name:8s} q={qs} c={cs} start={start:7.1f} ns finish={finish:7.1f} ns")

In [ ]:
qc_time = Circuit(2)
qc_time.h(0)
qc_time.cx(0, 1)
qc_time.barrier()
qc_time.x(1)
qc_time.cx(0, 1)

rows, duration = schedule_asap(qc_time)
print(qc_time.draw())
print_schedule(rows)
print("ASAP duration (ns):", duration)

In [ ]:
def pad_dynamical_decoupling_after_barriers(qc: Circuit) -> Circuit:
    # Toy dynamical-decoupling insertion: add X-X after a barrier on every qubit.
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_dd")

    for op in qc.operations:
        append_operation(out, op)
        if op.name == "barrier":
            for q in range(qc.n_qubits):
                out.x(q)
                out.x(q)

    return out


qc_idle = Circuit(1)
qc_idle.h(0)
qc_idle.barrier()
qc_idle.h(0)

qc_dd = pad_dynamical_decoupling_after_barriers(qc_idle)

print("Original:")
print(qc_idle.draw())
print_metrics("Original", qc_idle)

print("With toy DD padding:")
print(qc_dd.draw())
print_metrics("With DD", qc_dd)

print(
    "Unitary preserved up to global phase:",
    equal_up_to_global_phase(circuit_unitary(qc_idle), circuit_unitary(qc_dd))
)

# Subsection 8.5.4 **Logical resource estimation with circuit utilities**

In [ ]:
def add_t(qc, q):
    if hasattr(qc, "t"):
        return qc.t(q)
    return qc.unitary(qg.T, [q], name="t")


def add_tdg(qc, q):
    if hasattr(qc, "tdg"):
        return qc.tdg(q)
    return qc.unitary(qg.T.conj().T, [q], name="tdg")


def toffoli_t_template() -> Circuit:
    # Didactic template; not optimized for hardware.
    qc = Circuit(3, name="toffoli_template")
    qc.h(2)
    qc.cx(1, 2)
    add_tdg(qc, 2)
    qc.cx(0, 2)
    add_t(qc, 2)
    qc.cx(1, 2)
    add_tdg(qc, 2)
    qc.cx(0, 2)
    add_t(qc, 1)
    add_t(qc, 2)
    qc.h(2)
    return qc


template = toffoli_t_template()
print(template.draw())
print_metrics("Toffoli template", template)

In [ ]:
logical_gate_set = ["h", "s", "cx", "t", "tdg"]
print("Logical gate set:", logical_gate_set)

qc_t = Circuit(2)
qc_t.h(0)
add_t(qc_t, 0)
qc_t.cx(0, 1)
add_tdg(qc_t, 1)
add_t(qc_t, 1)

t_count = sum(1 for op in qc_t.operations if op.name in ["t", "tdg"])
print(qc_t.draw())
print("T-count:", t_count)

In [ ]:
class CancelOppositeT(TransformationPass):
    def run(self, qc: Circuit) -> Circuit:
        out_ops = []
        i = 0
        ops = qc.operations

        while i < len(ops):
            op = ops[i]
            if i + 1 < len(ops):
                nxt = ops[i + 1]
                if (
                    op.name in {"t", "tdg"}
                    and nxt.name in {"t", "tdg"}
                    and op.name != nxt.name
                    and tuple(op.targets) == tuple(nxt.targets)
                    and tuple(op.controls) == tuple(nxt.controls)
                ):
                    i += 2
                    continue
            out_ops.append(op)
            i += 1

        out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_no_t_pairs")
        for op in out_ops:
            append_operation(out, op)
        return out


qc_t_cancel = CancelOppositeT().run(qc_t)
print("Before:")
print(qc_t.draw())
print("After:")
print(qc_t_cancel.draw())
print_metrics("After T/Tdg cancellation", qc_t_cancel)

# Subsection 8.6.1 **Analysis and transformation passes**

In [ ]:
class BasisToRxRzCx(TransformationPass):
    def run(self, qc: Circuit) -> Circuit:
        return decompose_to_rx_rz_cx(qc)

class Optimize1qLocal(TransformationPass):
    def run(self, qc: Circuit) -> Circuit:
        return cancel_adjacent_gates(qc)


class TimingAnalysis(AnalysisPass):
    def run(self, qc: Circuit) -> Circuit:
        rows, duration = schedule_asap(qc)
        self.property_set["schedule"] = rows
        self.property_set["duration_ns"] = duration
        return qc


qc_pipe = Circuit(2)
qc_pipe.h(0)
qc_pipe.cx(0, 1)
qc_pipe.x(1)

pipeline = PassPipeline([
    BasisToRxRzCx(),
    Optimize1qLocal(),
    TimingAnalysis(),
])

optimized = pipeline.run(qc_pipe)
print(optimized.draw())
print("Pipeline properties:", pipeline.property_set)

# Subsection 8.6.2 **Equivalence-preserving rewrite libraries**

In [ ]:
class NonCliffordDensity(AnalysisPass):
    def run(self, qc: Circuit) -> Circuit:
        t_locations = []
        for i, op in enumerate(qc.operations):
            if op.name in ["t", "tdg"]:
                t_locations.append((i, op.targets))
        self.property_set["nonclifford_score"] = len(t_locations)
        self.property_set["t_locations"] = t_locations
        return qc


class SelectiveFuseT(TransformationPass):
    def run(self, qc: Circuit) -> Circuit:
        # Use the property set from a previous analysis pass if present.
        if self.property_set.get("nonclifford_score", 0) == 0:
            return qc
        return CancelOppositeT().run(qc)


qc_analysis = Circuit(1)
add_t(qc_analysis, 0)
add_tdg(qc_analysis, 0)
qc_analysis.h(0)

analysis_pipeline = PassPipeline([
    NonCliffordDensity(),
    SelectiveFuseT(),
])

qc_analysis_opt = analysis_pipeline.run(qc_analysis)
print("Before:")
print(qc_analysis.draw())
print("After:")
print(qc_analysis_opt.draw())
print("Property set:", analysis_pipeline.property_set)

# Subsection 8.6.3 **Combining global, algebraic, and timing-aware passes**

In [ ]:
# Register the identity CZ ≡ H CX H in the same simple equivalence library.
equiv_lib = SimpleEquivalenceLibrary()
equiv_lib.add_equivalence("cz", cz_to_hcxh)

qc_eq = Circuit(2)
qc_eq.cz(0, 1)
qc_eq.h(0)

pm_eq = PassPipeline([
    CZToCXRewrite(equiv_lib),
    BasisToRxRzCx(),
])

qc_eq_rewritten = pm_eq.run(qc_eq)
print("Original:")
print(qc_eq.draw())
print("After equivalence-library rewrite and basis decomposition:")
print(qc_eq_rewritten.draw())
print_metrics("Rewritten", qc_eq_rewritten)

# Subsection 8.6.4 **Complete qcirclab example**

In [ ]:
class FullOptimizationPipeline(PassPipeline):
    def __init__(self):
        super().__init__([
            NonCliffordDensity(),
            SelectiveFuseT(),
            CZToCXRewrite(equiv_lib),
            BasisToRxRzCx(),
            Optimize1qLocal(),
            GlobalCostAnalysis(alpha=1.0, beta=2.0),
            TimingAnalysis(),
        ])


qc_full = Circuit(3)
qc_full.h(0)
add_t(qc_full, 0)
qc_full.cx(0, 1)
add_tdg(qc_full, 1)
qc_full.cx(1, 2)
add_t(qc_full, 2)
qc_full.cz(0, 2)

full_pipeline = FullOptimizationPipeline()
optimized_full = full_pipeline.run(qc_full)

print("Original circuit:")
print(qc_full.draw())
print_metrics("Original", qc_full)

print("Optimized / compiled circuit:")
print(optimized_full.draw())
print_metrics("Optimized", optimized_full)

print("Property set:")
for k, v in full_pipeline.property_set.items():
    if k == "schedule":
        print(k, "=", len(v), "scheduled operations")
    else:
        print(k, "=", v)